# YOLOv8 Smart Parking — Kaggle GPU Training (50 epochs)

Trains **two** detectors on the full PKLot Roboflow export:

| Model | Classes | Purpose |
|---|---|---|
| `slot_det` | `space-empty`, `space-occupied` | Free / occupied parking slots |
| `car_det` | `car` (from occupied slots) | Approximate vehicle boxes |

**Wrong parking is NOT trained** — it is decided later by overlap geometry (`FREE` green / `CORRECT` blue / `WRONG` red).

## Before you run
1. Notebook settings → **Accelerator = GPU**, **Internet = On**
2. Add Input datasets:
   - PKLot YOLO export (`train/ valid/ test/ data.yaml`)
   - `parking-detection` code zip (`src/`, `configs/`, …)
3. Edit `CODE_INPUT` / `DATA_INPUT` in the next cell if names differ

**Training config:** `yolov8s`, **50 epochs**, batch 16, imgsz 640, patience 15, seed 42, full data (`--max-train 0`).

In [ ]:
%pip -q install ultralytics opencv-python-headless shapely pyyaml pandas matplotlib tqdm

from pathlib import Path
import torch

print('Inputs:', [p.name for p in Path('/kaggle/input').iterdir()])
assert torch.cuda.is_available(), 'Enable GPU: Settings → Accelerator → GPU, then Restart'
print('GPU:', torch.cuda.get_device_name(0))

# >>> EDIT these to match your attached dataset folder names <<<
CODE_INPUT = Path('/kaggle/input/parking-detection-code')
DATA_INPUT = Path('/kaggle/input/pklot-yolo-parking')

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

EPOCHS = 50
BATCH = 16
MODEL = 'yolov8s.pt'
IMGSZ = 640
PATIENCE = 15
SEED = 42
WORKERS = 2

def find_project_root(path: Path) -> Path:
    candidates = [path, *path.rglob('train.py')]
    for c in candidates:
        if c.name == 'train.py' and c.parent.name == 'src':
            return c.parent.parent
        if (c / 'src' / 'train.py').is_file():
            return c
    raise FileNotFoundError(f'Could not find src/train.py under {path}')

def find_dataset_root(path: Path) -> Path:
    for candidate in [path, *path.rglob('data.yaml')]:
        root = candidate.parent if candidate.name == 'data.yaml' else candidate
        if all((root / split / 'images').is_dir() and (root / split / 'labels').is_dir()
               for split in ('train', 'valid', 'test')):
            return root
    raise FileNotFoundError(f'Expected train/valid/test images+labels under {path}')

CODE_ROOT = find_project_root(CODE_INPUT)
RAW_DATA = find_dataset_root(DATA_INPUT)
PROJECT = Path('/kaggle/working/parking-detection')
PROJECT.mkdir(parents=True, exist_ok=True)

for folder in ('src', 'configs', 'tests'):
    src = CODE_ROOT / folder
    if src.is_dir():
        shutil.copytree(src, PROJECT / folder, dirs_exist_ok=True)

print('Code:', CODE_ROOT)
print('Dataset:', RAW_DATA)
print('Working:', PROJECT)
print(f'Train plan: model={MODEL} epochs={EPOCHS} batch={BATCH} imgsz={IMGSZ}')

def run(cmd):
    print('>>', ' '.join(map(str, cmd)))
    subprocess.run(cmd, cwd=PROJECT, check=True)

In [ ]:
# Prepare FULL datasets (no subsample). Original Roboflow files are never modified.
run([
    sys.executable, 'src/prepare_data.py',
    '--src', str(RAW_DATA),
    '--out', str(PROJECT / 'dataset'),
    '--max-train', '0',
    '--seed', str(SEED),
    '--copy',
])
print('space yaml:', (PROJECT / 'dataset' / 'space' / 'data.yaml').read_text()[:300])
print('car yaml:', (PROJECT / 'dataset' / 'car' / 'data.yaml').read_text()[:300])

In [ ]:
# Train SLOT detector — 50 epochs
run([
    sys.executable, 'src/train.py',
    '--data', str(PROJECT / 'dataset' / 'space' / 'data.yaml'),
    '--model', MODEL,
    '--name', 'slot_det',
    '--epochs', str(EPOCHS),
    '--batch', str(BATCH),
    '--imgsz', str(IMGSZ),
    '--device', '0',
    '--workers', str(WORKERS),
    '--patience', str(PATIENCE),
    '--seed', str(SEED),
    '--project', str(PROJECT / 'runs'),
])
slot_best = PROJECT / 'weights' / 'slot_det_best.pt'
assert slot_best.is_file(), slot_best
print('SLOT weights:', slot_best, 'size_MB', round(slot_best.stat().st_size / 1e6, 2))

In [ ]:
# Train CAR detector (occupied→car approximation) — 50 epochs
run([
    sys.executable, 'src/train.py',
    '--data', str(PROJECT / 'dataset' / 'car' / 'data.yaml'),
    '--model', MODEL,
    '--name', 'car_det',
    '--epochs', str(EPOCHS),
    '--batch', str(BATCH),
    '--imgsz', str(IMGSZ),
    '--device', '0',
    '--workers', str(WORKERS),
    '--patience', str(PATIENCE),
    '--seed', str(SEED),
    '--project', str(PROJECT / 'runs'),
])
car_best = PROJECT / 'weights' / 'car_det_best.pt'
assert car_best.is_file(), car_best
print('CAR weights:', car_best, 'size_MB', round(car_best.stat().st_size / 1e6, 2))

In [ ]:
# Evaluate on held-out TEST split (detection metrics only)
run([
    sys.executable, 'src/evaluation.py',
    '--weights', str(PROJECT / 'weights' / 'slot_det_best.pt'),
    '--data', str(PROJECT / 'dataset' / 'space' / 'data.yaml'),
    '--split', 'test',
    '--device', '0',
    '--out', str(PROJECT / 'outputs' / 'evaluation' / 'slot_metrics.json'),
    '--max-images', '100',
])
run([
    sys.executable, 'src/evaluation.py',
    '--weights', str(PROJECT / 'weights' / 'car_det_best.pt'),
    '--data', str(PROJECT / 'dataset' / 'car' / 'data.yaml'),
    '--split', 'test',
    '--device', '0',
    '--out', str(PROJECT / 'outputs' / 'evaluation' / 'car_metrics.json'),
    '--max-images', '100',
])
print('Done. Download from /kaggle/working/parking-detection/:')
print('  weights/slot_det_best.pt')
print('  weights/car_det_best.pt')
print('  runs/slot_det/results.csv')
print('  runs/car_det/results.csv')
print('  outputs/evaluation/*.json')

## After Kaggle — local inference

Copy the downloaded `.pt` files to:

`D:\RMS 4\parking-detection\weights\`

```powershell
cd "D:\RMS 4\parking-detection"
python src\inference.py `
  --source "D:\RMS 4\Parking_dataset (1)\Parking_dataset\test\images" `
  --slot-weights weights\slot_det_best.pt `
  --vehicle-weights weights\car_det_best.pt `
  --out outputs\inference\kaggle_50ep
```

If CUDA OOM during training: set `BATCH = 8` (or `4`) in the config cell and re-run the train cells.